# 03.01 — Bằng chứng Student Main-Core

Câu hỏi: Phase 3 đã freeze những Student nào để Phase 4 dùng? Trả lời dựa trên `main_core_decision.json`, metric validation và review đã khóa. Notebook chỉ đọc evidence; không tạo target, train Student, ghi metric, sửa review hoặc tạo artifact Phase 4.

## Kết quả đã khóa

Các giá trị dưới đây được ghi lại từ `main_core_decision.json`; các cell tiếp theo đọc lại đúng file này.

| Chiều | Run Main-Core | Checkpoint SHA-256 tree |
|---|---|---|
| EN→VI | `phase03-seqkd-en-to-vi-v2` | `c2f037f8548437bbd6edcaabc33b91f65c6fc7dcbd1124aa78239afe4b8e43bb` |
| VI→EN | `phase03-seqkd-vi-to-en-v3` | `fbcba2c1dbaa377073ab69893fabb51127c9de39609d1815f8fb97202442cb53` |

| Chiều | IT Validation: Student chrF++ | General Validation: Student chrF++ | Review 50 câu: lỗi dịch / lỗi kỹ thuật |
|---|---:|---:|---:|
| EN→VI | 66,55 | 52,44 | 28 / 19 |
| VI→EN | 61,30 | 46,20 | 24 / 19 |


In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'evidence/phase03/main_core_decision.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Không tìm thấy repository root hoặc Phase 3 decision.')

decision_path = ROOT / 'evidence/phase03/main_core_decision.json'
decision = json.loads(decision_path.read_text(encoding='utf-8'))
assert decision['status'] == 'completed_with_two_frozen_directions_for_phase04'

main_cores = []
for direction, item in decision['decisions'].items():
    main_cores.append({
        'chiều': direction,
        'run': item['run_id'],
        'Student': item['student']['model_id'],
        'checkpoint': item['checkpoint']['path'],
        'checkpoint SHA-256 tree': item['checkpoint']['sha256_tree'],
    })
pd.DataFrame(main_cores).sort_values('chiều')


,chiều,run,Student,checkpoint,checkpoint SHA-256 tree
1,en_to_vi,phase03-seqkd-en-to-vi-v2,Helsinki-NLP/opus-mt-en-vi,models/phase03/phase03-seqkd-en-to-vi-v2/final...,c2f037f8548437bbd6edcaabc33b91f65c6fc7dcbd1124...
0,vi_to_en,phase03-seqkd-vi-to-en-v3,Helsinki-NLP/opus-mt-vi-en,models/phase03/phase03-seqkd-vi-to-en-v3/final...,fbcba2c1dbaa377073ab69893fabb51127c9de39609d18...


In [2]:
rows = []
for direction, item in decision['decisions'].items():
    for dataset, evaluation in item['evaluation'].items():
        student = evaluation['student_metrics']
        teacher = evaluation['teacher_comparable_metrics']
        rows.append({
            'chiều': direction,
            'tập validation': dataset,
            'số câu': evaluation['rows'],
            'Student chrF++': student['chrF++'],
            'Teacher chrF++': teacher['chrF++'],
            'Student SacreBLEU': student['sacreBLEU'],
            'Teacher SacreBLEU': teacher['sacreBLEU'],
        })
validation = pd.DataFrame(rows).sort_values(['chiều', 'tập validation'])
validation


,chiều,tập validation,số câu,Student chrF++,Teacher chrF++,Student SacreBLEU,Teacher SacreBLEU
3,en_to_vi,general_validation,997,52.443036,60.364459,30.426077,39.238366
2,en_to_vi,selection_validation,15252,66.547210,71.179430,51.210707,57.756438
1,vi_to_en,general_validation,997,46.199546,58.449516,17.517299,29.842579
0,vi_to_en,selection_validation,15252,61.300553,65.339821,36.295198,41.229894


In [3]:
reviews = []
for direction, item in decision['decisions'].items():
    review = item['manual_review']
    reviews.append({
        'chiều': direction,
        'file review': review['path'],
        'số dòng': review['rows'],
        'lỗi dịch': review['translation_error_yes'],
        'lỗi kỹ thuật': review['technical_error_yes'],
    })
pd.DataFrame(reviews).sort_values('chiều')


,chiều,file review,số dòng,lỗi dịch,lỗi kỹ thuật
1,en_to_vi,reviews/phase03_phase03-seqkd-en-to-vi-v2_en_t...,50,28,19
0,vi_to_en,reviews/phase03_phase03-seqkd-vi-to-en-v3_vi_t...,50,24,19


## Student v1 so với baseline khởi tạo

Câu hỏi: sequence-level KD thay đổi Student v1 thế nào so với checkpoint OPUS-MT gốc? Hàm `read_student_baseline_comparison()` ở cell dưới chỉ đọc evidence đã khóa trên cùng cohort IT Validation 15.252 câu. Chênh lệch là `Student v1 - baseline`; không dùng kết quả này để đổi Main-Core.

In [4]:
def read_student_baseline_comparison(root: Path) -> pd.DataFrame:
    comparison_path = root / 'evidence/phase03/supplementary/phase03_student_v1_baseline_comparison_v1/comparison_result.json'
    comparison = json.loads(comparison_path.read_text(encoding='utf-8'))
    assert comparison['status'] == 'pass'

    rows = []
    for direction, result in comparison['results'].items():
        baseline = result['baseline']['metrics']
        student = result['student_v1']['metrics']
        delta = result['student_v1_minus_baseline']
        rows.append({
            'chiều': direction,
            'số câu': result['cohort_rows'],
            'baseline chrF++': baseline['chrF++'],
            'Student v1 chrF++': student['chrF++'],
            'chênh lệch chrF++': delta['chrF++'],
            'baseline SacreBLEU': baseline['sacreBLEU'],
            'Student v1 SacreBLEU': student['sacreBLEU'],
            'chênh lệch SacreBLEU': delta['sacreBLEU'],
        })
    return pd.DataFrame(rows).sort_values('chiều')

read_student_baseline_comparison(ROOT).round(2)


,chiều,số câu,baseline chrF++,Student v1 chrF++,chênh lệch chrF++,baseline SacreBLEU,Student v1 SacreBLEU,chênh lệch SacreBLEU
0,en_to_vi,15252,50.85,66.55,15.70,31.99,51.21,19.22
1,vi_to_en,15252,49.51,61.30,11.79,21.32,36.30,14.97


Kết quả trên là evidence của Phase 3. Student thấp hơn Teacher trên các cohort có thể so sánh; decision freeze không chứng minh Student ngang Teacher hoặc sẵn sàng chạy Android. Người thực hiện Phase 4 ở máy khác chỉ lấy checkpoint và hash từ decision theo `docs/phase_04/README.md`. Repository này không tạo artifact Phase 4.